In [3]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from sklearn.decomposition import NMF
import numpy as np
from nltk.corpus import stopwords
import spacy
from gensim.corpora import Dictionary
from gensim.models import LdaModel

## LDA 

In [5]:
# Path 
path = Path("../data/raw/text_files/1993/legislatives")
documents = []

# To get all .txt files in the folder and read their content
for file in path.glob("*.txt"):
    try:
        texte = file.read_text(encoding='utf-8', errors='ignore')
        documents.append({
            "filename": file.name,
            "texte_brut": texte})
    except Exception as e:
        print(f"Error with {file.name} : {e}")


df = pd.DataFrame(documents)
print(f"{len(df)} documents in the dataframe")

5936 documents in the dataframe


In [6]:
import pandas as pd

print(f"Tableau d'origine : {len(df)} documents complets.")

# 1. Découpage par sauts de ligne
# On utilise une expression régulière (r'\n+') pour couper le texte à chaque fois 
# qu'il y a un ou plusieurs sauts de ligne. Cela crée une liste de paragraphes.
df['liste_paragraphes'] = df['texte_brut'].str.split(r'\n+')

# 2. L'explosion du DataFrame (La magie de Pandas)
# La fonction explode() va prendre chaque paragraphe de la liste et créer une 
# nouvelle ligne dans le tableau, tout en dupliquant le 'filename' correspondant !
df_para = df.explode('liste_paragraphes').reset_index(drop=True)

# 3. On renomme la colonne pour que ce soit propre
df_para = df_para.rename(columns={'liste_paragraphes': 'paragraphe_brut'})

# 4. Le filtre anti-bruit (Crucial pour l'OCR)
# L'OCR génère souvent des "paragraphes" qui ne sont qu'un chiffre, un tiret 
# ou un mot coupé. On va éliminer tous les paragraphes de moins de 50 caractères.
df_para['paragraphe_brut'] = df_para['paragraphe_brut'].astype(str) # Sécurité
df_para = df_para[df_para['paragraphe_brut'].str.len() > 50].reset_index(drop=True)

print(f"Nouveau tableau : {len(df_para)} paragraphes robustes extraits !")
print("\n--- Aperçu des premiers paragraphes du premier fichier ---")
print(df_para[['filename', 'paragraphe_brut']].head(3))

Tableau d'origine : 5936 documents complets.
Nouveau tableau : 134600 paragraphes robustes extraits !

--- Aperçu des premiers paragraphes du premier fichier ---
                             filename  \
0  EL190_L_1993_03_016_02_2_PF_02.txt   
1  EL190_L_1993_03_016_02_2_PF_02.txt   
2  EL190_L_1993_03_016_02_2_PF_02.txt   

                                     paragraphe_brut  
0  République Française - Département de la Chare...  
1  JEAN-CLAUDE FAYEMENDIE Conseiller Municipal de...  
2  Alliance des Français pour le ProgrèsSciences ...  


In [8]:
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 16.7 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')


In [9]:
import spacy
from gensim.corpora import Dictionary
from gensim.models import LdaModel
import numpy as np

# --- ÉTAPE 1 : Nettoyage ultra-rapide par lots (Batch processing) ---
# On charge spaCy sans les fonctions lourdes
nlp = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

# On s'assure d'avoir notre liste de mots parasites
mots_parasites = {"candidat", "élection", "république", "français", "madame", "monsieur", "plus", "tout", "faire", "dire"}

print(f"Nettoyage de {len(df_para)} paragraphes en cours... (Patientez quelques minutes)")

# nlp.pipe() est la clé : il traite les textes par paquets de 1000, ce qui divise le temps par 10 !
textes_bruts = df_para['paragraphe_brut'].tolist()
mots_propres_list = []

for doc in nlp.pipe(textes_bruts, batch_size=1000):
    mots = [
        token.lemma_ for token in doc 
        if not token.is_punct and not token.is_space and not token.like_num 
        and not token.is_stop and len(token.lemma_) > 2 
        and token.lemma_ not in mots_parasites
    ]
    mots_propres_list.append(mots)

df_para['mots_propres'] = mots_propres_list

# Filtre de sécurité : on supprime les paragraphes qui se retrouvent vides après le nettoyage
df_para = df_para[df_para['mots_propres'].map(len) > 3].reset_index(drop=True)
print(f"Nettoyage terminé. {len(df_para)} paragraphes solides conservés.")

# --- ÉTAPE 2 : Modélisation LDA sur les paragraphes ---
textes_tokenises = df_para['mots_propres'].tolist()

dictionnaire = Dictionary(textes_tokenises)
# Note : On abaisse no_above à 0.5. Si un mot apparaît dans plus de 50% des paragraphes, c'est du bruit.
dictionnaire.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionnaire.doc2bow(texte) for texte in textes_tokenises]

print("\nEntraînement du modèle LDA en cours...")
nombre_de_themes = 15 # 15 thèmes donne assez d'espace pour isoler l'économie

lda_para = LdaModel(
    corpus=corpus,
    id2word=dictionnaire,
    num_topics=nombre_de_themes,
    random_state=42,
    passes=5 # 5 passages suffisent généralement pour des paragraphes
)

print("\n--- Les thèmes extraits ---")
for idx, topic in lda_para.print_topics(num_words=8):
    print(f"Thème {idx} : {topic}")


# --- ÉTAPE 3 : Agrégation (Reconstruire le document pour l'INSEE) ---
# 3a. Fonction pour trouver le thème numéro 1 de chaque paragraphe
def obtenir_theme_dominant(bow):
    if not bow: return -1
    topics = lda_para.get_document_topics(bow)
    return max(topics, key=lambda item: item[1])[0]

# On applique le thème à chaque ligne
df_para['theme_dominant'] = [obtenir_theme_dominant(bow) for bow in corpus]

# 3b. La magie de Pandas : Le tableau croisé pour avoir les % par document
# On compte combien de paragraphes de chaque thème il y a dans chaque fichier
theme_comptage = df_para.groupby(['filename', 'theme_dominant']).size().unstack(fill_value=0)

# On divise par le total de paragraphes du document pour avoir un pourcentage de 0 à 1
theme_pourcentage = theme_comptage.div(theme_comptage.sum(axis=1), axis=0).reset_index()

print("\n--- Pourcentage d'espace dédié à chaque thème par profession de foi ---")
print(theme_pourcentage.head())

Nettoyage de 134600 paragraphes en cours... (Patientez quelques minutes)
Nettoyage terminé. 130876 paragraphes solides conservés.

Entraînement du modèle LDA en cours...

--- Les thèmes extraits ---
Thème 0 : 0.025*"chômage" + 0.022*"immigration" + 0.020*"corruption" + 0.019*"génération" + 0.018*"entente" + 0.018*"insécurité" + 0.018*"chance" + 0.017*"injustice"
Thème 1 : 0.028*"droite" + 0.026*"France" + 0.026*"politique" + 0.024*"vouloir" + 0.018*"gauche" + 0.018*"force" + 0.016*"être" + 0.016*"pays"
Thème 2 : 0.018*"élu" + 0.016*"rétablir" + 0.015*"travail" + 0.014*"RPR" + 0.014*"UDF" + 0.014*"aujourd'hui" + 0.014*"payer" + 0.013*"compte"
Thème 3 : 0.065*"circonscription" + 0.060*"mars" + 0.031*"législatif" + 0.026*"député" + 0.021*"confiance" + 0.017*"tour" + 0.017*"parti" + 0.017*"national"
Thème 4 : 0.022*"vote" + 0.019*"meilleur" + 0.019*"trouver" + 0.017*"donner" + 0.016*"délinquant" + 0.016*"monde" + 0.016*"nom" + 0.015*"bulletin"
Thème 5 : 0.087*"fonds" + 0.054*"cevipof" + 0.

In [7]:
df_para.to_parquet("paragraphes_propres.parquet")


In [14]:
# --- ÉTAPE 2 : Modélisation LDA Guidée sur les paragraphes ---
textes_tokenises = df_para['mots_propres'].tolist()

dictionnaire = Dictionary(textes_tokenises)
dictionnaire.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionnaire.doc2bow(texte) for texte in textes_tokenises]

nombre_de_themes = 15
taille_vocabulaire = len(dictionnaire)

# 1. Création de la matrice Eta (les a priori)
# On met un poids de base ultra-faible (0.001) pour laisser les autres thèmes libres
eta_matrix = np.full((nombre_de_themes, taille_vocabulaire), 0.001)

# 2. Tes mots-graines (ceux validés par Word2Vec + les classiques)
mots_graines = ['chômage', 'inégalité', 'mage', 'chô-', 'chomage', 'grandissant', 'chomâge', 'croissant', 'fléau', 'grave', 'aggraver', 'précarité', 'subir', 'délinquance', 'intolérance', 'déficit', 'conséquence', 'creuser', 'difficulté', 'remède', 'marginalisation']

# 3. Le Forcing (On frappe très fort avec un poids de 10 000)
poids_extreme = 10000.0 
mots_trouves = 0

for mot in mots_graines:
    if mot in dictionnaire.token2id:
        mot_id = dictionnaire.token2id[mot]
        eta_matrix[0, mot_id] = poids_extreme # On force TOUT sur le Thème 0
        mots_trouves += 1

print(f"\nMatrice Eta configurée : {mots_trouves} mots-graines trouvés et injectés dans le Thème 0.")
print("Entraînement du modèle LDA Guidé en cours...")

# 4. Entraînement avec le paramètre magique 'eta'
lda_para = LdaModel(
    corpus=corpus,
    id2word=dictionnaire,
    num_topics=nombre_de_themes,
    eta=eta_matrix, # <--- C'est ici que l'on force le modèle
    random_state=42,
    passes=5
)

print("\n--- Les thèmes extraits ---")
for idx, topic in lda_para.print_topics(num_words=8):
    print(f"Thème {idx} : {topic}")


Matrice Eta configurée : 21 mots-graines trouvés et injectés dans le Thème 0.
Entraînement du modèle LDA Guidé en cours...

--- Les thèmes extraits ---
Thème 0 : 0.049*"difficulté" + 0.048*"conséquence" + 0.048*"chômage" + 0.047*"inégalité" + 0.047*"déficit" + 0.047*"délinquance" + 0.047*"aggraver" + 0.047*"grave"
Thème 1 : 0.029*"droite" + 0.027*"France" + 0.026*"politique" + 0.023*"vouloir" + 0.019*"gauche" + 0.018*"force" + 0.016*"pays" + 0.016*"être"
Thème 2 : 0.018*"élu" + 0.017*"RPR" + 0.016*"UDF" + 0.016*"rétablir" + 0.015*"travail" + 0.013*"aujourd'hui" + 0.013*"payer" + 0.013*"compte"
Thème 3 : 0.053*"circonscription" + 0.048*"mars" + 0.025*"législatif" + 0.022*"député" + 0.017*"confiance" + 0.017*"national" + 0.015*"parti" + 0.014*"tour"
Thème 4 : 0.032*"immigration" + 0.031*"chômage" + 0.026*"fiscal" + 0.025*"corruption" + 0.023*"insécurité" + 0.019*"combattre" + 0.018*"vote" + 0.016*"meilleur"
Thème 5 : 0.084*"fonds" + 0.053*"cevipof" + 0.041*"die" + 0.039*"science" + 0.03

In [4]:
# stopwords 
nltk.download('stopwords', quiet=True)
french_stopwords = stopwords.words('french')

# words related to manifestos

# Parameters
vectorizer = TfidfVectorizer(
    stop_words=french_stopwords,
    max_df=0.95,     # too many documents (95%+) contain this word, so it's not discriminant
    min_df=5,        # ignore if the word appears in less than 5 documents
    max_features=5000 # number of words  to keep based on term frequency
)

# Application
X_tfidf = vectorizer.fit_transform(df['texte_brut'])
print(f"Matrice créée : {X_tfidf.shape[0]} documents, {X_tfidf.shape[1]} mots de vocabulaire.")

Matrice créée : 5936 documents, 5000 mots de vocabulaire.


In [5]:
topic_number = 15

lda_model = LatentDirichletAllocation(n_components=topic_number, random_state=42)
W = lda_model.fit_transform(X_tfidf) # Remplacez X_tfidf par votre matrice de comptage si possible

# Afficher les mots principaux pour chaque topic
mots_du_vocabulaire = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda_model.components_):
    print(f"\nTopic {topic_idx + 1} :")
    top_mots_idx = topic.argsort()[:-11:-1]
    top_mots = [mots_du_vocabulaire[i] for i in top_mots_idx]
    print(" | ".join(top_mots))


Topic 1 :
indépendants | lacaze | cr | union | jeannou | armée | concrétisation | éléctions | renoncer | divisions

Topic 2 :
patrons | die | payer | patronat | und | der | maintenir | travailleurs | leurs | devons

Topic 3 :
12 | refus | instauration | ans | maintien | humiliation | mise | mixte | assez | populaire

Topic 4 :
voulons | paris | rôle | mr | reformateurs | réformateurs | mouvement | détruire | système | rapports

Topic 5 :
di | corsica | ch | in | una | da | so | corse | révolutionnaire | suivant

Topic 6 :
animaux | nature | nouveaux | rassemblement | existence | marseille | activités | ecologistes | doivent | homme

Topic 7 :
entente | écologie | ecologistes | verts | ecologie | generation | écologistes | chance | peu | génération

Topic 8 :
naturelle | cevipov | loi | animaux | associations | cohérence | nature | parti | universaliste | bénévole

Topic 9 :
français | suppression | front | sauvons | mains | françaises | défense | allègement | suspension | apprentissag

In [6]:
# Most dominant topic for each document
df['theme_dominant'] = np.argmax(W, axis=1)
print(df[['filename', 'theme_dominant']].head(10))

                             filename  theme_dominant
0  EL190_L_1993_03_016_02_2_PF_02.txt              13
1  EL189_L_1993_03_003_02_1_PF_01.txt              13
2  EL193_L_1993_03_063_01_2_PF_01.txt              13
3  EL196_L_1993_03_076_12_1_PF_08.txt               5
4  EL190_L_1993_03_030_05_2_PF_01.txt              13
5  EL192_L_1993_03_051_04_2_PF_01.txt              13
6  EL192_L_1993_03_044_04_2_PF_01.txt              13
7  EL190_L_1993_03_022_02_1_PF_06.txt               5
8  EL198_L_1993_03_092_09_1_PF_03.txt              13
9  EL192_L_1993_03_056_03_1_PF_06.txt               5


## Guided LDA for analysis on unemployment

In [7]:
# French language model from spacy
nlp = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

# Cleaning function to extract proper words from the raw text
def obtenir_mots_propres(texte):
    if not isinstance(texte, str):
        return []
    doc = nlp(texte.lower())
    mots_propres = []
    
    for token in doc:
        if not token.is_punct and not token.is_space and not token.like_num and not token.is_stop:
            lemme = token.lemma_
            if lemme not in french_stopwords and len(lemme) > 2:
                mots_propres.append(lemme)
                
    return mots_propres


df['mots_propres'] = df['texte_brut'].apply(obtenir_mots_propres)
print(df[['texte_brut', 'mots_propres']].head())

                                          texte_brut  \
0  Sciences Po / fonds CEVIPOF\nRépublique França...   
1  ENTENTE DES ECOLOGISTES\nÉlections législative...   
2  DÉPARTEMENT DU PUY-DE-DÔME\nRÉPUBLIQUE FRANÇAI...   
3  RÉPUBLIQUE FRANÇAISE - ÉLECTIONS LÉGISLATIVES ...   
4  DANILET ALAIN\nSUPPLEANT : Christian BURGLÉ (P...   

                                        mots_propres  
0  [science, fonds, cevipof, république, français...  
1  [entente, ecologiste, élection, législatif, sc...  
2  [département, puy-de-dôme, république, françai...  
3  [république, français, élection, législatif, m...  
4  [danilet, alain, suppleant, christian, burgler...  


In [8]:
textes_tokenises = df['mots_propres'].tolist() 

# Dictionary and Corpus for Gensim
dictionnaire = Dictionary(textes_tokenises)

# Filtering on the dictionary to remove very rare and very common words
dictionnaire.filter_extremes(no_below=5, no_above=0.95)
corpus = [dictionnaire.doc2bow(texte) for texte in textes_tokenises]

nombre_de_themes = 5
taille_vocabulaire = len(dictionnaire)

eta_matrix = np.full((nombre_de_themes, taille_vocabulaire), 0.01) #parameter eta low  

# Words to force into the topic of unemployment
mots_graines = ['chômage', 'mage', 'chô-', 'inégalité', 'chomage', 'chomâge', 'délinquance', 'inéga-', 'grandissant', 'grave', 'précarité', 'décourager', 'insupportable', 'profiter', 'subir', 'conséquence', 'aggraver', 'croissant', 'incontrôlé', 'intolérance', 'briser']
# More weight for the seed words in the first topic (index 0)
for mot in mots_graines:
    if mot in dictionnaire.token2id:
        mot_id = dictionnaire.token2id[mot]
        eta_matrix[0, mot_id] = 100 #parameter eta high for the seed words


# Training
lda_guide = LdaModel(
    corpus=corpus,
    id2word=dictionnaire,
    num_topics=nombre_de_themes,
    eta=eta_matrix, 
    random_state=42,
    passes=10 # Number of passes through the corpus during training (more passes can improve results but take more time)
)

# Results
print("\n Topics extracted from the model :")
for idx, topic in lda_guide.print_topics(num_words=10):
    print(f"Topic {idx} : {topic}\n")


 Topics extracted from the model :
Topic 0 : 0.022*"nature" + 0.017*"nouveau" + 0.016*"animal" + 0.012*"rassemblement" + 0.012*"candidat" + 0.011*"naturel" + 0.009*"loi" + 0.008*"nom" + 0.008*"ecologiste" + 0.008*"programme"

Topic 1 : 0.019*"faire" + 0.013*"écologie" + 0.013*"travailleur" + 0.012*"entente" + 0.012*"entreprise" + 0.012*"payer" + 0.011*"emploi" + 0.011*"ecologiste" + 0.011*"maintenir" + 0.011*"patron"

Topic 2 : 0.029*"die" + 0.024*"der" + 0.021*"und" + 0.019*"communiste" + 0.013*"droite" + 0.011*"faire" + 0.010*"vote" + 0.008*"parti" + 0.007*"für" + 0.007*"den"

Topic 3 : 0.010*"politique" + 0.009*"candidat" + 0.008*"social" + 0.007*"circonscription" + 0.006*"emploi" + 0.006*"france" + 0.005*"faire" + 0.005*"national" + 0.005*"vie" + 0.005*"pays"

Topic 4 : 0.035*"français" + 0.018*"national" + 0.015*"rpr" + 0.015*"udf" + 0.013*"voter" + 0.012*"immigration" + 0.012*"france" + 0.010*"vouloir" + 0.010*"impôt" + 0.009*"front"



In [9]:
from gensim.models import Word2Vec
textes_tokenises = df['mots_propres'].tolist()
modele_w2v = Word2Vec(sentences=textes_tokenises, vector_size=100, window=5, min_count=5, workers=4)
mot_cible = 'chômage' 

if mot_cible in modele_w2v.wv.key_to_index:
    mots_similaires = modele_w2v.wv.most_similar(mot_cible, topn=20)
    
    print(f"\nVoici le champ lexical exact autour de '{mot_cible}' ")
    mots_trouves = []
    
    for mot, score in mots_similaires:
        print(f"- {mot} (Similarité: {score:.2f})")
        mots_trouves.append(mot)
        
    print(f"mots_graines = {['chômage'] + mots_trouves}")
else:
    print(f"Le mot '{mot_cible}' n'a pas été trouvé dans le vocabulaire.")


Voici le champ lexical exact autour de 'chômage' 
- inégalité (Similarité: 0.64)
- mage (Similarité: 0.62)
- chô- (Similarité: 0.59)
- chomage (Similarité: 0.53)
- grandissant (Similarité: 0.53)
- chomâge (Similarité: 0.50)
- croissant (Similarité: 0.50)
- fléau (Similarité: 0.48)
- grave (Similarité: 0.48)
- aggraver (Similarité: 0.47)
- précarité (Similarité: 0.47)
- subir (Similarité: 0.46)
- délinquance (Similarité: 0.46)
- intolérance (Similarité: 0.46)
- déficit (Similarité: 0.46)
- conséquence (Similarité: 0.45)
- creuser (Similarité: 0.45)
- difficulté (Similarité: 0.45)
- remède (Similarité: 0.44)
- marginalisation (Similarité: 0.44)
mots_graines = ['chômage', 'inégalité', 'mage', 'chô-', 'chomage', 'grandissant', 'chomâge', 'croissant', 'fléau', 'grave', 'aggraver', 'précarité', 'subir', 'délinquance', 'intolérance', 'déficit', 'conséquence', 'creuser', 'difficulté', 'remède', 'marginalisation']


## BERTopic

In [4]:
from bertopic import BERTopic

/opt/miniconda3/envs/mon_projet_nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print(f"Préparation de {len(df_para)} paragraphes pour BERTopic...")

# 1. Préparation des textes (BERTopic veut des chaînes de caractères, pas des listes)
def nettoyage_robuste(valeur):
    if isinstance(valeur, list):
        return " ".join(valeur)
    elif isinstance(valeur, str):
        return valeur.replace("[", "").replace("]", "").replace("'", "").replace(",", "")
    return ""

docs = df_para['mots_propres'].apply(nettoyage_robuste).tolist()

# 2. Configuration du Zero-Shot Topic Modeling
# Tu décris le thème exactement comme tu en parlerais à un humain
themes_cibles = [
    "chômage, emploi, précarité, licenciement et crise économique"
]

print("Chargement du modèle d'Intelligence Artificielle...")
topic_model = BERTopic(
    embedding_model="paraphrase-multilingual-MiniLM-L12-v2",
    zeroshot_topic_list=themes_cibles,
    zeroshot_min_similarity=0.80, # On baisse un tout petit peu la sévérité pour attraper plus de paragraphes
    language="french",
    verbose=True
)

# 3. Entraînement et Extraction
print("Entraînement en cours (création des embeddings, UMAP, HDBSCAN)...")
topics, probs = topic_model.fit_transform(docs)

# On injecte les résultats dans notre tableau de paragraphes
df_para['topic_bertopic'] = topics

# On affiche ce que le modèle a trouvé
print("\n--- Les thèmes extraits par BERTopic ---")
print(topic_model.get_topic_info().head(10))

# 4. AGRÉGATION : La reconstruction des professions de foi pour l'INSEE
print("\n--- Calcul du poids du chômage par profession de foi ---")

# On fait un tableau croisé dynamique : on compte combien de paragraphes de chaque thème il y a par fichier
theme_comptage = df_para.groupby(['filename', 'topic_bertopic']).size().unstack(fill_value=0)

# On divise par le nombre total de paragraphes du document pour obtenir un pourcentage
theme_pourcentage = theme_comptage.div(theme_comptage.sum(axis=1), axis=0).reset_index()

# Le thème que l'on a forcé s'appellera exactement par la chaîne de caractères qu'on a définie
# ou bien il prendra l'index 0. Affiche les colonnes pour vérifier son nom exact :
print("Aperçu des scores thématiques par document :")
print(theme_pourcentage.head())

ModuleNotFoundError: No module named 'bertopic'